[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/22_conv2d_solution.ipynb)

# Solution: 2D Convolution

Reference solution.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# ✅ SOLUTION

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    B, C_in, H, W = x.shape
    C_out, _, kH, kW = weight.shape

    H_out = (H + 2 * padding - kH) // stride + 1
    W_out = (W + 2 * padding - kW) // stride + 1

    # Extract sliding windows: [B, C_in*kH*kW, H_out*W_out]
    patches = F.unfold(
        x, kernel_size=(kH, kW),
        stride=stride, padding=padding
    )

    # Flatten kernels: [C_out, C_in*kH*kW]
    weight_flat = weight.reshape(C_out, -1)

    # Matrix multiplication: [B, C_out, H_out*W_out]
    out = weight_flat @ patches

    # Reshape back to spatial dimensions
    out = out.reshape(B, C_out, H_out, W_out)

    if bias is not None:
        out += bias.view(1, -1, 1, 1)

    return out

In [ ]:
# Demo
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

In [ ]:
from torch_judge import check
check('conv2d')